# Football API Check

## Imports

In [1]:
import sys, pathlib
_root = pathlib.Path.cwd()
while not (_root / "src").exists() and _root != _root.parent:
    _root = _root.parent
sys.path.insert(0, str(_root))

import warnings
warnings.filterwarnings("ignore")

import json
import os
import urllib.request

import pandas as pd

from src.worldcup import config as C
from src.worldcup.live import WC_FILE, fetch_results


## Is the football-data.org API reachable and authenticated?

In [2]:
def api_status(timeout=15):
    key = os.getenv("FOOTBALL_DATA_API_KEY", "")
    if not key:
        return "no key set (FOOTBALL_DATA_API_KEY); using cached snapshot"
    req = urllib.request.Request("https://api.football-data.org/v4/competitions/WC/matches",
                                 headers={"X-Auth-Token": key})
    try:
        with urllib.request.urlopen(req, timeout=timeout) as r:
            return "reachable: HTTP %d" % r.status
    except Exception as e:
        return "request failed (%s); using cached snapshot" % e

print(api_status())
print("cached file:", WC_FILE.name, "exists:", WC_FILE.exists())


no key set (FOOTBALL_DATA_API_KEY); using cached snapshot
cached file: WC_2026_matches.json exists: True


## FIFA World Cup 2026 results have been played so far

In [3]:
raw = json.load(open(WC_FILE))
matches = raw["matches"]
finished = [m for m in matches if m["status"] == "FINISHED"]
played = pd.DataFrame([{
    "date": m["utcDate"][:10],
    "stage": m["stage"].replace("_", " ").title(),
    "home": m["homeTeam"]["name"], "away": m["awayTeam"]["name"],
    "score": "%s-%s" % (m["score"]["fullTime"]["home"], m["score"]["fullTime"]["away"]),
} for m in finished])
print("played matches:", len(finished))
played


played matches: 2


,date,stage,home,away,score
0,2026-06-11,Group Stage,Mexico,South Africa,2-0
1,2026-06-12,Group Stage,South Korea,Czechia,2-1


## Upcoming World Cup fixtures

In [4]:
upcoming = [m for m in matches if m["status"] != "FINISHED"]
upcoming = sorted(upcoming, key=lambda m: m["utcDate"])[:10]
fixtures = pd.DataFrame([{
    "date": m["utcDate"][:10],
    "stage": m["stage"].replace("_", " ").title(),
    "group": (m.get("group") or "").replace("GROUP_", "") or "-",
    "home": m["homeTeam"]["name"], "away": m["awayTeam"]["name"],
} for m in upcoming])
print("scheduled matches remaining:", sum(m["status"] != "FINISHED" for m in matches))
fixtures


scheduled matches remaining: 102


,date,stage,group,home,away
0,2026-06-12,Group Stage,B,Canada,Bosnia-Herzegovina
1,2026-06-13,Group Stage,D,United States,Paraguay
2,2026-06-13,Group Stage,B,Qatar,Switzerland
3,2026-06-13,Group Stage,C,Brazil,Morocco
4,2026-06-14,Group Stage,C,Haiti,Scotland
5,2026-06-14,Group Stage,D,Australia,Turkey
6,2026-06-14,Group Stage,E,Germany,Curaçao
7,2026-06-14,Group Stage,F,Netherlands,Japan
8,2026-06-14,Group Stage,E,Ivory Coast,Ecuador
9,2026-06-15,Group Stage,F,Sweden,Tunisia


## Fields the API exposes for each match

In [5]:
m = matches[0]
print("match fields:", list(m.keys()))
print("score fields:", list(m["score"].keys()))
print("team fields: ", list(m["homeTeam"].keys()))
print()
print("pre-kickoff (usable as features): utcDate, stage, group, matchday, homeTeam, awayTeam, odds")
print("post-match (target / leak):       score.fullTime, score.winner")


match fields: ['area', 'competition', 'season', 'id', 'utcDate', 'status', 'matchday', 'stage', 'group', 'lastUpdated', 'homeTeam', 'awayTeam', 'score', 'odds', 'referees']
score fields: ['winner', 'duration', 'fullTime', 'halfTime']
team fields:  ['id', 'name', 'shortName', 'tla', 'crest']

pre-kickoff (usable as features): utcDate, stage, group, matchday, homeTeam, awayTeam, odds
post-match (target / leak):       score.fullTime, score.winner


## Available squad data per team

In [6]:
teams = json.load(open(C.RAW / "footballdata_api" / "WC_teams.json"))["teams"]
with_squad = [t for t in teams if t.get("squad")]
print("teams:", len(teams), " with a named squad:", len(with_squad))
print("team fields: ", list(teams[0].keys()))
print("player fields:", list(with_squad[0]["squad"][0].keys()))
fetch_check = fetch_results(refresh=False)
print("\nfetch_results() returns", len(fetch_check), "finished matches in canonical-name form")


teams: 48  with a named squad: 48
team fields:  ['area', 'id', 'name', 'shortName', 'tla', 'crest', 'address', 'website', 'founded', 'clubColors', 'venue', 'runningCompetitions', 'coach', 'squad', 'staff', 'lastUpdated']
player fields: ['id', 'name', 'position', 'dateOfBirth', 'nationality']

fetch_results() returns 2 finished matches in canonical-name form
